# Tokenização na Prática
### Do texto ao número: como modelos de linguagem leem o mundo

---

Antes de qualquer modelo de linguagem processar uma frase, ele precisa converter texto em números.  
Esse processo se chama **tokenização** — e entender como ele funciona muda a forma como você pensa em LLMs.

Neste notebook vamos explorar:

1. O algoritmo **BPE (Byte Pair Encoding)** — como ele é treinado
2. O tokenizador **tiktoken** (OpenAI / GPT) na prática
3. O tokenizador **BERT Multilingual** (HuggingFace / WordPiece)
4. Um experimento comparativo: **PT-BR vs EN** — qual língua é mais eficiente?

---

## 0. Instalação das dependências

In [1]:
pip install tiktoken transformers pandas -q


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. O Algoritmo BPE por Dentro

O **Byte Pair Encoding** é o algoritmo usado pelo GPT para construir seu vocabulário.  
A ideia central é simples: *fundir os pares de caracteres mais frequentes, repetidamente.*

A biblioteca `tiktoken._educational` expõe uma implementação didática desse processo.
Vamos usá-la para ver o BPE funcionando passo a passo — exatamente como acontece durante o treinamento do tokenizador.

In [5]:
import tiktoken._educational as edu

bpe = edu.SimpleBytePairEncoding.from_tiktoken("gpt2")

# Veja todos os atributos disponíveis
print(dir(bpe))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_decoder', '_pat', 'decode', 'decode_bytes', 'decode_tokens_bytes', 'encode', 'from_tiktoken', 'mergeable_ranks', 'pat_str', 'train']


In [6]:
import tiktoken._educational as edu

bpe = edu.SimpleBytePairEncoding.from_tiktoken("gpt2")

# Ordena por rank (ordem de criação durante o treinamento)
tokens_por_rank = sorted(bpe.mergeable_ranks.items(), key=lambda x: x[1])

print("=== Primeiros 30 tokens por ordem de aprendizado (rank) ===")
print("(rank baixo = token mais fundamental, aprendido primeiro)")
print()

for token_bytes, rank in tokens_por_rank[:30]:
    try:
        token_str = token_bytes.decode("utf-8")
    except UnicodeDecodeError:
        token_str = repr(token_bytes)  # bytes não-UTF8 (ex: caracteres especiais)
    
    n_chars = len(token_bytes)
    fase = "caractere único" if n_chars == 1 else f"merge ({n_chars} chars)"
    
    print(f"Rank {rank:05d} | {repr(token_str):25s} | {fase}")

=== Primeiros 30 tokens por ordem de aprendizado (rank) ===
(rank baixo = token mais fundamental, aprendido primeiro)

Rank 00000 | '!'                       | caractere único
Rank 00001 | '"'                       | caractere único
Rank 00002 | '#'                       | caractere único
Rank 00003 | '$'                       | caractere único
Rank 00004 | '%'                       | caractere único
Rank 00005 | '&'                       | caractere único
Rank 00006 | "'"                       | caractere único
Rank 00007 | '('                       | caractere único
Rank 00008 | ')'                       | caractere único
Rank 00009 | '*'                       | caractere único
Rank 00010 | '+'                       | caractere único
Rank 00011 | ','                       | caractere único
Rank 00012 | '-'                       | caractere único
Rank 00013 | '.'                       | caractere único
Rank 00014 | '/'                       | caractere único
Rank 00015 | '0'          

In [7]:
# Filtra só tokens de 1 caractere e vê quais são
tokens_simples = [
    (token_bytes.decode("utf-8", errors="replace"), rank)
    for token_bytes, rank in tokens_por_rank
    if len(token_bytes) == 1
]

print("Tokens de 1 caractere ordenados por rank:")
for char, rank in tokens_simples[:50]:
    print(f"  Rank {rank:04d} → {repr(char)}")

Tokens de 1 caractere ordenados por rank:
  Rank 0000 → '!'
  Rank 0001 → '"'
  Rank 0002 → '#'
  Rank 0003 → '$'
  Rank 0004 → '%'
  Rank 0005 → '&'
  Rank 0006 → "'"
  Rank 0007 → '('
  Rank 0008 → ')'
  Rank 0009 → '*'
  Rank 0010 → '+'
  Rank 0011 → ','
  Rank 0012 → '-'
  Rank 0013 → '.'
  Rank 0014 → '/'
  Rank 0015 → '0'
  Rank 0016 → '1'
  Rank 0017 → '2'
  Rank 0018 → '3'
  Rank 0019 → '4'
  Rank 0020 → '5'
  Rank 0021 → '6'
  Rank 0022 → '7'
  Rank 0023 → '8'
  Rank 0024 → '9'
  Rank 0025 → ':'
  Rank 0026 → ';'
  Rank 0027 → '<'
  Rank 0028 → '='
  Rank 0029 → '>'
  Rank 0030 → '?'
  Rank 0031 → '@'
  Rank 0032 → 'A'
  Rank 0033 → 'B'
  Rank 0034 → 'C'
  Rank 0035 → 'D'
  Rank 0036 → 'E'
  Rank 0037 → 'F'
  Rank 0038 → 'G'
  Rank 0039 → 'H'
  Rank 0040 → 'I'
  Rank 0041 → 'J'
  Rank 0042 → 'K'
  Rank 0043 → 'L'
  Rank 0044 → 'M'
  Rank 0045 → 'N'
  Rank 0046 → 'O'
  Rank 0047 → 'P'
  Rank 0048 → 'Q'
  Rank 0049 → 'R'


### 1.1 As 3 fases do BPE

Quando o BPE processa a palavra `"tokenization"`, ele segue exatamente estas etapas:

| Fase | O que acontece | Exemplo |
|------|---------------|--------|
| **1 — Separação** | Cada caractere vira um token individual | `t o k e n i z a t i o n` |
| **2 — Merges iterativos** | Pares mais frequentes são fundidos | `to ke ni za ti on` → ... |
| **3 — Token final** | Palavra comum vira 1 único token | `tokenization` |

Vamos ver isso acontecer visualmente:

In [8]:
import tiktoken._educational as edu

bpe = edu.SimpleBytePairEncoding.from_tiktoken("gpt2")

palavras = ["tokenization", "tokenização", "cat", "gato"]

print("=== Visualização das fases do BPE ===\n")
for palavra in palavras:
    tokens = bpe.encode(palavra)
    pedacos = [bpe.decode([t]) for t in tokens]
    print(f"Palavra  : '{palavra}'")
    print(f"Fase 1   : {list(palavra)}")
    print(f"Resultado: {pedacos}")
    print(f"Nº tokens: {len(tokens)}")
    print("-" * 50)

=== Visualização das fases do BPE ===

tokenization
tokenization
tokenization
tokenization
tokenization
tokenization
tokenization
tokenization
tokenization
tokenization
tokenization

Palavra  : 'tokenization'
Fase 1   : ['t', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 't', 'i', 'o', 'n']
Resultado: ['token', 'ization']
Nº tokens: 2
--------------------------------------------------
tokeniza����o
tokeniza����o
tokeniza����o
tokeniza����o
tokeniza����o
tokenizaç��o
tokenizaç��o
tokenização
tokenização
tokenização

Palavra  : 'tokenização'
Fase 1   : ['t', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 'ç', 'ã', 'o']
Resultado: ['token', 'iza', 'ç', 'ão']
Nº tokens: 4
--------------------------------------------------
cat
cat
cat

Palavra  : 'cat'
Fase 1   : ['c', 'a', 't']
Resultado: ['cat']
Nº tokens: 1
--------------------------------------------------
gato
gato
gato

Palavra  : 'gato'
Fase 1   : ['g', 'a', 't', 'o']
Resultado: ['g', 'ato']
Nº tokens: 2
--------------------------------------------------


**O que observar aqui:**
- `cat` provavelmente vira **1 token** — palavra muito frequente em inglês no corpus de treino
- `gato` provavelmente vira **2+ tokens** — corpus é majoritariamente em inglês
- `tokenization` vs `tokenização` — compare e anote a diferença

Isso vai fazer mais sentido no experimento PT vs EN da seção 4.

## 2. tiktoken na Prática

Agora vamos usar o `tiktoken` da forma que você usaria num projeto real — direto com o tokenizador do **GPT-4**.

In [9]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4")

texto = "Natural language processing enables machines to understand human language."

tokens = enc.encode(texto)
pedacos = [enc.decode([t]) for t in tokens]

print(f"Texto original : {texto}")
print(f"Nº de tokens   : {len(tokens)}")
print(f"IDs numéricos  : {tokens}")
print(f"Pedaços         : {pedacos}")

Texto original : Natural language processing enables machines to understand human language.
Nº de tokens   : 10
IDs numéricos  : [55381, 4221, 8863, 20682, 12933, 311, 3619, 3823, 4221, 13]
Pedaços         : ['Natural', ' language', ' processing', ' enables', ' machines', ' to', ' understand', ' human', ' language', '.']


### 2.1 Uma observação importante: espaços fazem parte dos tokens

Note que alguns tokens começam com espaço (ex: `' language'`).  
O BPE do GPT trata espaço como parte do token — isso é uma decisão de design que afeta como o modelo entende fronteiras entre palavras.

In [10]:
# Visualizando quais tokens incluem espaço
print("Token | Tem espaço? | ID")
print("-" * 35)
for token_id, pedaco in zip(tokens, pedacos):
    tem_espaco = "✓" if pedaco.startswith(" ") else " "
    print(f"{repr(pedaco):20s} | {tem_espaco:11s} | {token_id}")

Token | Tem espaço? | ID
-----------------------------------
'Natural'            |             | 55381
' language'          | ✓           | 4221
' processing'        | ✓           | 8863
' enables'           | ✓           | 20682
' machines'          | ✓           | 12933
' to'                | ✓           | 311
' understand'        | ✓           | 3619
' human'             | ✓           | 3823
' language'          | ✓           | 4221
'.'                  |             | 13


## 3. BERT Multilingual — O Algoritmo WordPiece

O BERT usa um algoritmo diferente: **WordPiece**.  
A diferença principal para o BPE é como os merges são escolhidos:  
- BPE: funde o par **mais frequente** 
- WordPiece: funde o par que **mais aumenta a probabilidade do corpus**

Na prática, a diferença mais visível é o símbolo `##` — que indica que aquele pedaço é uma **continuação** de uma palavra, não o início dela.

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

texto = "Natural language processing enables machines to understand human language."

output = tokenizer(texto)
tokens_bert = tokenizer.convert_ids_to_tokens(output["input_ids"])

print(f"Texto original : {texto}")
print(f"Nº de tokens   : {len(tokens_bert)}")
print(f"Tokens          : {tokens_bert}")

c:\Users\oguis\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
c:\Users\oguis\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\oguis\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/

Texto original : Natural language processing enables machines to understand human language.
Nº de tokens   : 13
Tokens          : ['[CLS]', 'Natural', 'language', 'processing', 'enable', '##s', 'machines', 'to', 'understand', 'human', 'language', '.', '[SEP]']


**Tokens especiais do BERT:**
- `[CLS]` — marcador de início de sequência
- `[SEP]` — marcador de separação / fim
- `##` — indica continuação de subpalavra

O GPT não tem esses tokens especiais porque foi treinado de forma diferente (geração vs classificação).

## 4. Experimento: PT-BR vs EN

**Hipótese:** textos em português consomem mais tokens que o equivalente em inglês, porque o corpus de treinamento dos tokenizadores é majoritariamente em inglês — então menos merges de palavras portuguesas foram aprendidos.

Vamos testar essa hipótese com frases equivalentes.

In [12]:
import tiktoken
from transformers import AutoTokenizer
import pandas as pd

enc_gpt4 = tiktoken.encoding_for_model("gpt-4")
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

pares = [
    ("Natural language processing", "Processamento de linguagem natural"),
    ("Machine learning is powerful", "Aprendizado de máquina é poderoso"),
    ("The cat sat on the mat", "O gato sentou no tapete"),
    ("Artificial intelligence", "Inteligência artificial"),
    ("Deep learning neural networks", "Redes neurais de aprendizado profundo"),
]

resultados = []

for en, pt in pares:
    gpt_en = len(enc_gpt4.encode(en))
    gpt_pt = len(enc_gpt4.encode(pt))
    bert_en = len(tokenizer_bert.encode(en, add_special_tokens=False))
    bert_pt = len(tokenizer_bert.encode(pt, add_special_tokens=False))
    
    resultados.append({
        "Frase EN": en,
        "Frase PT": pt,
        "GPT-4 (EN)": gpt_en,
        "GPT-4 (PT)": gpt_pt,
        "GPT-4 diferença": f"+{gpt_pt - gpt_en} tokens",
        "BERT (EN)": bert_en,
        "BERT (PT)": bert_pt,
        "BERT diferença": f"+{bert_pt - bert_en} tokens",
    })

df = pd.DataFrame(resultados)

# Exibe as colunas de resultado
colunas_resultado = ["Frase EN", "Frase PT", "GPT-4 (EN)", "GPT-4 (PT)", "GPT-4 diferença", "BERT (EN)", "BERT (PT)", "BERT diferença"]
pd.set_option("display.max_colwidth", 40)
df[colunas_resultado]

,Frase EN,Frase PT,GPT-4 (EN),GPT-4 (PT),GPT-4 diferença,BERT (EN),BERT (PT),BERT diferença
0,Natural language processing,Processamento de linguagem natural,3,6,+3 tokens,3,5,+2 tokens
1,Machine learning is powerful,Aprendizado de máquina é poderoso,4,9,+5 tokens,4,7,+3 tokens
2,The cat sat on the mat,O gato sentou no tapete,6,8,+2 tokens,6,8,+2 tokens
3,Artificial intelligence,Inteligência artificial,3,4,+1 tokens,3,4,+1 tokens
4,Deep learning neural networks,Redes neurais de aprendizado profundo,4,9,+5 tokens,5,9,+4 tokens


### 4.1 Calculando o overhead médio

In [13]:
total_gpt_en = sum(r["GPT-4 (EN)"] for r in resultados)
total_gpt_pt = sum(r["GPT-4 (PT)"] for r in resultados)
total_bert_en = sum(r["BERT (EN)"] for r in resultados)
total_bert_pt = sum(r["BERT (PT)"] for r in resultados)

overhead_gpt = ((total_gpt_pt - total_gpt_en) / total_gpt_en) * 100
overhead_bert = ((total_bert_pt - total_bert_en) / total_bert_en) * 100

print("=== Overhead de tokens: PT-BR vs EN ===")
print()
print(f"GPT-4  → EN: {total_gpt_en} tokens | PT: {total_gpt_pt} tokens | Overhead: +{overhead_gpt:.1f}%")
print(f"BERT   → EN: {total_bert_en} tokens | PT: {total_bert_pt} tokens | Overhead: +{overhead_bert:.1f}%")
print()
print("Conclusão: textos em PT-BR consomem mais tokens — o que impacta custo de API e tamanho do contexto.")

=== Overhead de tokens: PT-BR vs EN ===

GPT-4  → EN: 20 tokens | PT: 36 tokens | Overhead: +80.0%
BERT   → EN: 21 tokens | PT: 33 tokens | Overhead: +57.1%

Conclusão: textos em PT-BR consomem mais tokens — o que impacta custo de API e tamanho do contexto.


## 5. Conclusões

---

### O que aprendemos neste notebook:

**Sobre o BPE:**
- O algoritmo parte de caracteres individuais e funde pares frequentes iterativamente
- As regras de merge são aprendidas durante o treinamento do tokenizador — não do modelo
- Palavras comuns viram 1 token; palavras raras ou em idiomas sub-representados ficam fragmentadas

**Sobre os dois tokenizadores:**

| Característica | tiktoken (GPT) | BERT Multilingual |
|---|---|---|
| Algoritmo | BPE | WordPiece |
| Tokens especiais | Nenhum | `[CLS]`, `[SEP]` |
| Espaço | Parte do token | Separado |
| Continuação | Não sinalizada | Prefixo `##` |

**Sobre PT-BR vs EN:**
- PT-BR consome mais tokens que EN para conteúdo equivalente
- Isso tem impacto direto em **custo de API** e **tamanho do contexto disponível**
- É um dado importante ao projetar aplicações LLM para o mercado brasileiro

---

### Próximos passos sugeridos:
- **Temperatura** — como o modelo escolhe entre os tokens possíveis após a tokenização
- **Embeddings** — como os IDs de tokens viram vetores com significado semântico